# main追加評価：通常条件の量拡大 ＋ A/Bsの空間拡張（Windows）
既存30kモデルを固定し、**①元条件の確認評価 → ②広い空間でのA/Bs比較**を一つのNotebookで実施します。新規学習はしません。

- ① **A・B75・B50・E**：既存確認bankから各N25形状、計100形状。学習3方式×3seed＋Eで **1,000 episode**。Eは各形状1回です。
- ② **A・B75・B50のみ**：低CPAの交差航路を後方へ延ばした**広い空間1条件だけ**。各N10形状で **360 episode**。0/20/40秒の条件比較はしません。

既定の合計は**1,360 episode**。試走は本実行で再利用するので追加分ではありません。
両者のbank・保存先・集計・zipを分離します。①は通常分布の確認、②は低CPAを多く含む空間拡張の追加ストレス評価です。相互に混ぜて平均しません。
①の比較方式・指標は開始時に固定します。結果を見て再調整したモデルは同じ確認結果に混在させません。

②では交差領域・ゴールを保ち、出発点を航路沿いに後方へ延ばします。距離の指定には「初速×40秒」を内部的に用いますが、時間別の実験群は作りません。実際の空間寸法と接近猶予は保存した初期条件・軌跡から確認できます。

In [ ]:
from pathlib import Path
import sys, json
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'pyproject.toml').exists())
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))
from tqdm.notebook import tqdm
from scripts.main_route_extension_runner import prepare, checkpoints, select_scenarios, evaluate_grid, status, export_results
from scripts import main_confirmation_runner as confirmation
from uav_safe_marl.evaluation import load_scenario_bank
from uav_safe_marl.runtime import runtime_manifest
SOURCE = ROOT / 'results' / 'main_additional_20261120'
OUTPUT = ROOT / 'results' / 'main_spatial_extension_single_20261127'
BANK_PER_N = 100
EVAL_PER_N = 10
COUNTS = (2, 4, 8, 16)
LEADS = (40.,)  # 広い空間1条件のみ。延長量の比較はしない
SEEDS = (0, 1, 2)
EVALUATION_DEVICE = 'cpu'
RUN_PROBE = True
RUN_FULL = True
RUN_EXPORT = True

# ①通常条件：既存の未使用確認bankを固定して量を拡大
CONFIRM_OUTPUT = ROOT / 'results' / 'main_confirmation_20261127'
CONFIRM_BANK_PER_N = 100
CONFIRM_EVAL_PER_N = 25
RUN_CONFIRMATION = True
# ②空間拡張（1条件）：以下の既存OUTPUT/BANK_PER_N/EVAL_PER_Nを使用
RUN_EXTENSION = True
# RUN_PROBE / RUN_FULL / RUN_EXPORTは両実験に共通


## ① 通常条件：固定モデルの確認評価
元mainと同一の四方交通・初期間隔30m・予測horizon50秒・終了上限を使います。
SOURCE/banks/confirmation_test.jsonのhashを元実験と照合し、初期状態を変更せず使用します。CPAの小さい形状だけを選びません。

N別を主とし、全N等重み平均を併記します。初期CPA別の接近距離・接近率、介入時間/強度/Cost、QP可解性/fallback、到着時間/経路/timeout、通信を解析対象として固定します。Eの同一結果を3seed分の独立試行として数えません。

CONFIRM_EVAL_PER_Nだけを減らせば段階実行できます。初回は各N25形状、計100形状・1,000episodeです。元の分析48形状から独立形状数を約2倍に増やします。必要なら50、100へ増やして未完了分だけ追加できます。bank自体は各N100形状で固定し、結果を見て形状を選び直しません。

In [ ]:
if RUN_CONFIRMATION:
    confirm_experiment = confirmation.prepare(SOURCE, CONFIRM_OUTPUT, per_n=CONFIRM_BANK_PER_N, counts=COUNTS)
    confirm_ids = confirmation.select_scenarios(CONFIRM_OUTPUT, CONFIRM_EVAL_PER_N)
    print('通常条件の今回の評価量:', len(confirm_ids) * (3 * len(SEEDS) + 1), 'episode')
    print('保存先:', CONFIRM_OUTPUT)
    confirm_models = confirmation.checkpoints(CONFIRM_OUTPUT, SEEDS)
    for (method, seed), path in confirm_models.items(): print(f'{method}/{seed}: {path}')
    print('選択した評価デバイス:', EVALUATION_DEVICE)
    print(json.dumps(runtime_manifest(), ensure_ascii=False, indent=2))

### 通常条件の確認実走と本実行
最小/最大Nの共通2形状をseed0で4方式、計8case試走し、そのまま本実行に進みます。
case数と実step数をtqdm表示し、直列で1モデルずつ処理します。完了caseは本実行・再開時に再利用します。
RUN_FULL=Falseなら両実験とも試走だけ、RUN_EXTENSION=Falseなら通常条件だけを実行します。

In [ ]:
if RUN_CONFIRMATION:
    confirm_bank = load_scenario_bank(CONFIRM_OUTPUT / 'banks' / 'confirmation_test.json')
    confirm_probe_ids = [next(s['scenario_id'] for s in confirm_bank if s['scenario_id'] in set(confirm_ids) and s['actual_agent_count'] == n)
                         for n in sorted({min(COUNTS), max(COUNTS)})]
    if RUN_PROBE:
        confirm_probe = confirmation.evaluate_grid(CONFIRM_OUTPUT, seeds=[SEEDS[0]], device=EVALUATION_DEVICE,
                                                   scenario_ids=confirm_probe_ids, progress_factory=tqdm)
        print('通常条件・確認済みcase:', confirm_probe['completed_cases'])
    if RUN_FULL:
        confirm_result = confirmation.evaluate_grid(CONFIRM_OUTPUT, seeds=SEEDS, device=EVALUATION_DEVICE,
                                                    scenario_ids=confirm_ids, progress_factory=tqdm)
        print('通常条件の今回の完了:', confirm_result['requested_completed_cases'], '/', confirm_result['requested_cases'])

### 通常条件の途中集計・Mac持ち帰り
生NPZ・全指標・bank・固定解析方針・使用モデル・hashを保存します。途中停止後もこのセルだけで取り出せます。
通常条件と空間拡張は別zipです。こちらを保存した後、下の空間拡張へ進みます。

In [ ]:
if RUN_CONFIRMATION:
    confirm_partial = confirmation.status(CONFIRM_OUTPUT)
    print(json.dumps(confirm_partial, ensure_ascii=False, indent=2))
    if RUN_EXPORT:
        confirm_archive = confirmation.export_results(CONFIRM_OUTPUT, include_models=True)
        print('通常条件の持ち帰りzip:', confirm_archive)
        print(f'サイズ: {confirm_archive.stat().st_size / 1024**2:.1f} MiB')

## ② 空間拡張：広い空間1条件の共通bank
BANK_PER_N=100で各N100形状の拡張bankを固定します。実行はEVAL_PER_N=10の計40形状、3方式×3seedで **360episode**。
狭い元空間や中間の延長量は評価しません。全形状で同じ延長規則を適用した広い空間だけを比較します。

低CPAの遭遇を保つため、開始位置を初速×40秒分だけ後方へ延ばします。予測horizonは90秒、終了上限も長い航路に合わせる従来の拡張仕様です。これは空間拡張に付随する評価設定であり、時間別の実験群ではありません。

EVAL_PER_Nだけなら同じOUTPUTで評価量を増やせます。BANK_PER_N/延長量/COUNTS/生成条件を変える場合は別OUTPUTへ保存します。
旧0/20/40秒版と混ざらないよう、保存先をmain_spatial_extension_single_20261127に分けています。

In [ ]:
if RUN_EXTENSION:
    experiment = prepare(SOURCE, OUTPUT, per_n=BANK_PER_N, counts=COUNTS, leads=LEADS)
    print(f"元形状: {experiment['base_geometries']} / 延長後シナリオ: {experiment['scenarios']}")
    print(f"全量: {experiment['scenarios'] * 3 * len(SEEDS):,} episode / 最大step: {experiment['max_steps']:,}")
    selected_ids = select_scenarios(OUTPUT, EVAL_PER_N)
    print('今回の評価量:', len(selected_ids) * 3 * len(SEEDS), 'episode')
    print('生成試行数:', experiment['generation_draws'])
    print('保存先:', OUTPUT)


## 評価モデルとデバイス
既存の親実験A/B、追加実験B50、またはSOURCE/modelsの正規名モデルを利用します。不足時は必要なパスを表示して停止します。
CPU評価を明示しています。安全QPもCPUで動作します。CUDAへ変える場合は開始前に設定し、同じ保存先の途中で変更しないでください。


In [ ]:
if RUN_EXTENSION:
    models = checkpoints(OUTPUT, SEEDS)
    for (method, seed), path in models.items(): print(f'{method}/{seed}: {path}')
    print('選択した評価デバイス:', EVALUATION_DEVICE)
    print(json.dumps(runtime_manifest(), ensure_ascii=False, indent=2))


## 短い実走確認
最小/最大Nの共通2シナリオをseed0の3方式、計6caseで実行し、保存・所要時間を確認します。
モデルとhashを検証し、case進捗と飛行中の実step数を表示します。
完了caseは全量評価で再利用します。RUN_FULL=Falseにすれば確認だけで止まります。

In [ ]:
if RUN_EXTENSION:
    bank = load_scenario_bank(OUTPUT / 'banks' / 'route_extension.json')
    probe_ids = []
    for n in sorted({min(COUNTS), max(COUNTS)}):
        for lead in sorted({min(LEADS), max(LEADS)}):
            probe_ids.append(next(s['scenario_id'] for s in bank if s['actual_agent_count'] == n and s['route_extension_seconds'] == lead))
    if RUN_PROBE:
        probe = evaluate_grid(OUTPUT, seeds=[SEEDS[0]], device=EVALUATION_DEVICE, scenario_ids=probe_ids, progress_factory=tqdm)
        print('確認済みcase:', probe['completed_cases'])


## 全量評価
1条件ずつ直列で実行し、並列workerは使いません。学習・Replay保存は行いません。
中断時に飛行中だった1caseはやり直し、完了markerと生NPZが揃うcaseは再実行しません。


In [ ]:
if RUN_EXTENSION:
    if RUN_FULL:
        result = evaluate_grid(OUTPUT, seeds=SEEDS, device=EVALUATION_DEVICE, scenario_ids=selected_ids, progress_factory=tqdm)
        print('今回の完了:', result['requested_completed_cases'], '/', result['requested_cases'])
        print('全bankで完了:', result['completed_cases'], '/', result['expected_all_seeds_cases'])


## 途中集計・取り出し
このセルは途中停止後にも実行できます。全episodeのJSON/CSV、生NPZ、bank、初期条件、出典、モデルhashを保持します。
集計の違反率は記録点での判定です。後から区間内最近接・接近ペア分布・Actor急変・可解性・通信離脱も解析できます。
成功限定の経路比/完了時間は、timeoutを含むepisode時間と分けて集計します。


In [ ]:
if RUN_EXTENSION:
    partial = status(OUTPUT)
    print(json.dumps(partial, ensure_ascii=False, indent=2))


## Macへ持ち帰るzip
既定でA/B75/B50の評価用モデルも含めます（Replay・再開用checkpoint・runtime一時ファイルは除外）。
zip内のexport_manifest.jsonに全ファイルのhashとbyte数を記録します。保存先は評価フォルダの外です。
このセルだけ途中で実行すれば、その時点の完了結果を取り出せます。


In [ ]:
if RUN_EXTENSION:
    if RUN_EXPORT:
        archive = export_results(OUTPUT, include_models=True)
        print('持ち帰りzip:', archive)
        print(f'サイズ: {archive.stat().st_size / 1024**2:.1f} MiB')
